# RegVelo

https://github.com/theislab/cellrank/issues/592

In [648]:
import regvelo as rgv
import scvelo as scv
import cellrank as cr

import scanpy as sc

import pickle

from scipy import sparse
import pandas as pd
import numpy as np
import math

import matplotlib.pyplot as plt

import seaborn as sns
import matplotlib.pyplot as plt

import os

## Settings

In [649]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

In [650]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning) 

# Warnings scanpy off
sc.settings.verbosity = 0

## Plotting 

In [651]:
# Scanpy figures
cm_to_inch = lambda x: x / 2.54
sc.set_figure_params(figsize=(cm_to_inch(10), cm_to_inch(10)), dpi_save=300, fontsize=16, frameon=False, color_map='Reds', facecolor='white')

# Load pre-processed RNA velocity data

In [652]:
from cellrank.kernels import VelocityKernel
vk = VelocityKernel.read(fname='data/object/cellrank/kernel/vk_0.pickle')

In [653]:
adata = vk.adata.copy()

# Load GRN

In [630]:
# load saved regulon-target matrix
reg = pd.read_csv("result/scenic/res_0/adj.csv", index_col=0)

In [631]:
# Transform regulon table
reg = reg.reset_index().pivot_table(
    
    index="TF",
    columns="target",
    values="importance",
    aggfunc="sum", 
    fill_value=0

)

In [632]:
reg.index = reg.index.str.extract(r"(\w+)")[0]
reg = reg.groupby(reg.index).sum()

In [633]:
# Binarize
reg[reg!=0] = 1

In [634]:
TF = np.unique(list(map(lambda x: x.split("(")[0], reg.index.tolist())))
genes = np.unique(TF.tolist() + reg.columns.tolist())

In [635]:
GRN = pd.DataFrame(0, index=genes, columns=genes)
GRN.loc[TF,reg.columns.tolist()] = np.array(reg)

In [636]:
mask = (GRN.sum(0) != 0) | (GRN.sum(1) != 0)
GRN = GRN.loc[mask, mask].copy()

In [637]:
target_candidate = reg[tf_candidate]
target_candidate = target_candidate[target_candidate.sum(axis=1)>0].index.to_list()

In [638]:
set(target_candidate) - set(GRN)

set()

# Preprocess

In [639]:
adata = rgv.pp.set_prior_grn(adata, GRN.T)

In [643]:
set(target_candidate) - set(adata.uns['skeleton'])

{'ASCL2',
 'ATF1',
 'ATF6B',
 'ATF7',
 'BLZF1',
 'CARHSP1',
 'CCDC88A',
 'CDIP1',
 'CEBPD',
 'CEBPG',
 'CENPX',
 'CREBZF',
 'DDIT3',
 'DEAF1',
 'DNAJC2',
 'E2F7',
 'EOMES',
 'ESRRA',
 'FOSL1',
 'FOXK1',
 'FOXM1',
 'GCFC2',
 'GMEB1',
 'HIVEP3',
 'HMGA1',
 'ID3',
 'IRF9',
 'JUN',
 'JUNB',
 'KLF10',
 'KLF12',
 'KLF13',
 'KLF4',
 'KLF5',
 'MAF1',
 'MAFF',
 'MAFG',
 'MBD3',
 'MBD4',
 'MEF2A',
 'MGA',
 'MLLT10',
 'MLX',
 'MSC',
 'MTA1',
 'MXD4',
 'MYC',
 'MYSM1',
 'NFYC',
 'NR1D1',
 'NR1H2',
 'NR2C1',
 'PLEK',
 'PMS1',
 'POU2F1',
 'RARG',
 'RBCK1',
 'RELA',
 'RFX2',
 'RFX5',
 'RORC',
 'SETBP1',
 'SMAD4',
 'SMAD7',
 'SMARCC2',
 'SMARCE1',
 'SNAI3',
 'SP1',
 'SP140L',
 'SP2',
 'SREBF1',
 'SSRP1',
 'STAT2',
 'TCF19',
 'TGIF2',
 'TP53',
 'TP63',
 'TTF1',
 'TULP3',
 'XPA',
 'YEATS4',
 'ZBTB11',
 'ZBTB16',
 'ZBTB44',
 'ZFPM1',
 'ZFX',
 'ZFY',
 'ZHX1',
 'ZMIZ2',
 'ZNF101',
 'ZNF107',
 'ZNF143',
 'ZNF160',
 'ZNF22',
 'ZNF280D',
 'ZNF296',
 'ZNF326',
 'ZNF33A',
 'ZNF414',
 'ZNF429',
 'ZNF430',
 'ZNF4

In [644]:
velocity_genes = rgv.pp.preprocess_data(adata.copy()).var_names.tolist()

# select TFs that regulate at least one gene
TF = adata.var_names[adata.uns["skeleton"].sum(axis=1) != 0]
var_mask = np.union1d(TF, velocity_genes)

adata = adata[:, var_mask].copy()

# update skeleton matrix
adata.uns["skeleton"] = adata.uns["skeleton"].loc[adata.var_names.tolist(), adata.var_names.tolist()]

computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)


In [645]:
genes_tmp = tmp[tmp.sum(axis=1)>0].index.to_list()

In [647]:
set(tf_candidate) - set(TF)

{'RORC'}

In [ ]:
adata = rgv.pp.filter_genes(adata)

# perform min-max scaling
adata = rgv.pp.preprocess_data(adata, filter_on_r2=False)

adata.var["velocity_genes"] = adata.var_names.isin(velocity_genes)
adata.var["TF"] = adata.var_names.isin(TF)

## GRN velocity 

In [ ]:
import torch
from regvelo import REGVELOVI

In [ ]:
vae_cache = True 

In [ ]:
if vae_cache: 
    
    vae = REGVELOVI.load('data/object/regvelo/model_0', adata)

else: 
    
    W = adata.uns["skeleton"].copy()
    W = torch.tensor(np.array(W)).int()
    W = W.T
    
    REGVELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
    
    vae = REGVELOVI(adata, W=W, regulators=TF)
    
    vae.train(max_epochs=1500, early_stopping=True)
    
    vae.save("data/object/regvelo/model_0", overwrite=True)

In [ ]:
def model_history(model): 
    
    # Plot model history 
    fig, axes=plt.subplots(1, 2, figsize=(10, 5))
    
    axes[0].plot(model.history['reconstruction_loss_train']['reconstruction_loss_train'], label='train')
    axes[0].plot(model.history['reconstruction_loss_validation']['reconstruction_loss_validation'], label='validation')
    axes[0].set_title('Reconstruction Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    
    axes[1].plot(model.history['elbo_train']['elbo_train'], label='train')
    axes[1].plot(model.history['elbo_validation']['elbo_validation'], label='validation')
    axes[1].set_title('ELBO')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

In [ ]:
model_history(vae)

In [ ]:
cache_regvelo = True 

In [ ]:
if not cache_regvelo: 
    
    # Add GRN information to adata
    rgv.tl.set_output(adata, vae, n_samples=30, batch_size=adata.n_obs)
    
    # Save 
    adata.write_h5ad('data/object/regvelo/adata_0.h5ad')

else: 

    adata = sc.read_h5ad('data/object/regvelo/adata_0.h5ad')

# Recompute embedding 

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep='latent')
sc.tl.umap(adata, min_dist=1)

# CellRank model based on RegVelo

## Compute cell type labels based on module score

In [ ]:
# Add module scores
ms = pd.read_csv('result/ms/ms.csv', index_col=0)
adata.obs = adata.obs.merge(ms, left_index=True, right_index=True, how='left')

## Velocity and GRN kernel

In [ ]:
from cellrank.kernels import VelocityKernel
vk = VelocityKernel.read(fname='data/object/cellrank/kernel/vk_0.pickle')
vk = vk.compute_transition_matrix()

## AUC Kernel

In [ ]:
from cellrank.kernels import ConnectivityKernel
ck = ConnectivityKernel.read(fname='data/object/cellrank/kernel/ck_0.pickle')
ck = ck.compute_transition_matrix()

## Combine Kernel

In [ ]:
k = 0.5*vk + 0.5*ck

In [ ]:
# Use RegVelo AnnData
k.adata = adata.copy()

## Estimator 

In [ ]:
g = cr.estimators.GPCCA(k)

In [ ]:
g.compute_macrostates(n_states=12, n_cells=50, cluster_key="celltype_low")

## Terminal macrostates

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Prepare and aggregate the data (calculate mean score per macrostate)
df = g.adata.obs[['macrostates_fwd', 'TRM17_vs_TRM1_ucell', 'TRM17_ucell', 'TRM1_vs_TRM17_ucell', 'TRM1_ucell']].copy()
df = df.dropna(subset=['macrostates_fwd'])

# Group by macrostate and calculate the mean for each column
df_mean = df.groupby('macrostates_fwd').mean()

# 2. Plot the heatmap
plt.figure(figsize=(8, 8))
sns.heatmap(
    df_mean, 
    annot=True,             # Show the actual values inside the cells
    cmap='viridis',         # Color scheme ('coolwarm', 'Blues', 'RdBu_r' are also good)
    fmt='.3f',              # Round numbers to 3 decimal places
    linewidths=0.5,         # Add a thin line between cells
    cbar_kws={'label': 'Mean Score'} # Label for the colorbar on the right
)

plt.title('Mean UCell Scores by Macrostate')
plt.ylabel('Macrostates')
plt.xlabel('Score Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
g.plot_macrostates(which='all', basis='X_umap', discrete=True, legend_loc='on data', figsize=(6, 6), size=100)

In [ ]:
tmp = adata.raw.to_adata()
sc.pp.normalize_total(tmp)
sc.pp.log1p(tmp)

sc.pl.umap(tmp, color=['time_point', 'S_ucell', 'G2M_ucell', 'TRM1_vs_TRM17_ucell', 'TRM17_vs_TRM1_ucell', 'TRM1_ucell', 'TRM17_ucell', 
                       'CD4', 'CD8A', 'CD8B',
                       'IFNG', 'GZMA', 'GZMB',  
                       'CD69', 'PRDM1', 
                       'CD3E', 'CD3D', 'CD3G', 'CD247', 'TRAC', 'TRBC1', 'TRBC2'], 
           size=50, use_raw=False, ncols=7)

In [ ]:
terminal_states = [
    
    '$T_{c}\ RM$_1', # Low PRDM1 and no activation signature 
    '$T_{c}\ RM$_10, $T_{c}\ RM$_11', # T17
    '$T_{c}\ RM$_2, $T_{c}\ RM$_7' #T1 
    
]

## Set terminal states and compute fate probabilities

In [ ]:
g.set_terminal_states(terminal_states, cluster_key="celltype_low")

In [ ]:
g.compute_fate_probabilities(solver="direct")

In [ ]:
g.plot_fate_probabilities(same_plot=False, basis="umap")

In [ ]:
cr.pl.circular_projection(g.adata, keys=['celltype_low', 'time_point', 'hto_demux_class', 'TRM1_vs_TRM17_ucell', 'TRM17_vs_TRM1_ucell'], legend_loc="right")

# TF perturbation in the regulatory network

In [ ]:
tf_candidate = ['MAF', 'RORC', 'TBX21', 'ZNF683']

In [ ]:
adata_perturb_dict = {}
for TF in tf_candidate:
    adata_perturb, reg_vae_perturb =  rgv.tl.in_silico_block_simulation(model='data/object/regvelo/model_0', 
                                                                        adata=g.adata, 
                                                                        TF=TF, 
                                                                        cutoff=0)
    adata_perturb_dict[TF] = adata_perturb

In [ ]:
ct_indices = {
    ct: adata.obs["term_states_fwd"][adata.obs["term_states_fwd"] == ct].index.tolist()
    for ct in terminal_states}